# Study on Tool Calling by agents

## Prepare

In [5]:
import langchain
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.tools import tool
from langchain.agents import create_agent

print(f'Import LangChain V{langchain.__version__}')

Import LangChain V1.2.7


In [15]:
with open('api-key/deepseek.txt', 'r') as f:
    llm = ChatOpenAI(
        base_url="https://api.deepseek.com",
        api_key=f.read().strip(),
        model="deepseek-v4-flash",
        temperature=0,
        extra_body={"thinking": {"type": "disabled"}}
    )

# print(
#     StrOutputParser().invoke(
#         llm.invoke([("human", "Introduce yourself briefly")])
#     )
# )

## Tool demo

In [38]:
@tool
def search_information(query: str) -> str:
    """
    provide info due to given query.
    answer questions such as 'capital of france', 'weather in london', etc.
    use English to query.
    """
    print(f'=== Try to query "{query}" by tool ===')
    simulated_results = {
        'weather in london': "London is cloudy rightnow, temperature 15°C。",
        "capital of france": "Capital of France is Paris",
        "population of earth": "The global population is about 8 billion",
        "tallest mountain": "珠穆朗玛峰是海拔最高的山峰。",
        "default": f"模拟搜索 '{query}'：未找到具体信息，但该主题很有趣。"
    }
    result = simulated_results.get(query.lower(), simulated_results["default"])
    print(f"Got result {result}")
    return result

## Build agent using tools

In [39]:
agent = create_agent(llm, [search_information],
                     system_prompt="你是一个乐于助人的助手。")

## Test

In [33]:
questions = [
    "What's capatal of france",
    "How is the weather of london",
    "Tell me some info about dogs"
]

In [41]:
for qust in questions:
    print(f'Question: {qust}')
    response = await agent.ainvoke(
        {"messages": [{"role": "user", "content": qust}]})
    answer = response["messages"][1].content
    print(f'Answer: {answer}')

Question: What's capatal of france
Answer: The capital of France is **Paris**.
Question: How is the weather of london
=== Try to query "weather in London" by tool ===
Got result London is cloudy rightnow, temperature 15°C。
Answer: Let me check the current weather in London for you.
Question: Tell me some info about dogs
Answer: Here's some interesting information about dogs!

## 🐕 Basic Facts About Dogs

**Scientific name:** *Canis lupus familiaris*
**Lifespan:** 10–13 years (varies by breed)
**Origin:** Domesticated from gray wolves around 15,000–40,000 years ago

## 🧠 Key Characteristics

- **Highly social animals** – They're pack animals that thrive on companionship
- **Incredible sense of smell** – Up to 100,000 times more sensitive than humans
- **Hearing** – Can hear sounds up to 4 times farther away than humans
- **Communication** – Use barking, whining, growling, body language, and tail wagging

## 🐶 Popular Dog Breeds

| Breed | Known For |
|-------|-----------|
| Labrador Ret